In [21]:
from aux import *
import matplotlib.pyplot as plt

df = pd.read_pickle("data_family.pkl")
exog = df['exog'].copy().ffill()
sales_by_family = df['sales_by_family'] 
familias = pd.read_csv("familias.csv")
data_by_family = {
    fam: format_as_year_month(sales_by_family[sales_by_family['family'] == fam].copy())
    for fam in sales_by_family['family'].unique()
}
#for fam in data_by_family:
#    data_by_family[fam], _ = limpiar_outliers_x_agrupacion(data_by_family[fam], 'family', 'sale_amount_MM')
name = '0421'
target_col = 'sale_amount_MM'
ignore_col = [] 
negatives_reg_col = []
feat = feature_selection(data_by_family[name], exog, max_lag=4, min_lag=-3,
                         target_col=target_col, ignore_col=ignore_col,
                         negatives_reg_col=negatives_reg_col)
feat = feat[feat['correlación'] > 0.4].reset_index(drop=True)
feat = clean_focus_correlation(feat, group='variable', focus='correlación')
feat = feat.sort_values(by='correlación', ascending=False).reset_index(drop=True)
selected, resumen = collinearity_analysis(data_by_family[name], exog, feat, target_col)
feat = feat[feat['variable'].isin(selected)]
df_final = construir_dataset_familia(name, data_by_family, exog, feat, target_col)
splits = generar_splits(df_final)

/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/statsmodels/tsa/stattools.py:1179: RuntimeWarning: invalid value encountered in divide
  ret = cvf / (np.std(x) * np.std(y))


In [22]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_family_sales(data_by_family, familias, names, fig_width=18, fig_height=10):
    rows = len(names)
    fig, axes = plt.subplots(rows, 1, figsize=(fig_width, fig_height), sharex=True,
                             constrained_layout=True) 
    if rows == 1:
        axes = [axes]
    for idx, name in enumerate(names):
        serie = data_by_family[name].sale_amount_MM
        x = serie.index
        y = serie.values
        ax = axes[idx]
        ax.plot(x, y, color='steelblue', linewidth=2)
        family_name = familias.loc[familias.hier_family_cd == int(name), 'hier_family_name'].values[0]
        ax.set_title(f"Evolución de Ventas Mensuales - {family_name}", fontsize=13)
        ax.set_ylabel("Ventas (MM)", fontsize=11)
        ax.grid(True, linestyle='--', alpha=0.5)
        if x.dtype.kind in {'M', 'm'}:
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
            ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
            ax.tick_params(axis='x', rotation=45)
    axes[-1].set_xlabel("Fecha", fontsize=12)
    plt.show()
names = [ '0105', '0421','0312']  # lista de códigos de familia que quieres graficar
plot_family_sales(data_by_family, familias, names)

/var/folders/_2/b07hnlxx27j566j7xpz5w05m0000gn/T/ipykernel_67521/2740982820.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, RepeatVector, TimeDistributed
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler

def create_seq2seq_data(X, y, window_size, n_ahead):
    Xs, ys = [], []
    for i in range(window_size, len(X) - n_ahead + 1):
        Xs.append(X[i - window_size:i])
        ys.append(y[i:i + n_ahead])
    return np.array(Xs), np.array(ys)

def fit_predict_eval_seq2seq_lstm(training_set, test_set, model_params=None):
    """
    Entrena y evalúa un modelo LSTM tipo Seq2Seq con múltiples pasos de salida.
    """
    default_params = {
        'window_size': 12,
        'n_ahead': 6,
        'units': 64,
        'epochs': 50,
        'batch_size': 16,
        'learning_rate': 0.001
    }
    if model_params:
        default_params.update(model_params)
    p = default_params

    # Escalado
    features = training_set.drop(columns=['ds', 'y']).columns
    scaler_x = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_train = scaler_x.fit_transform(training_set[features])
    y_train = scaler_y.fit_transform(training_set[['y']])
    X_test = scaler_x.transform(test_set[features])
    y_test = scaler_y.transform(test_set[['y']])

    # Datos para entrenamiento
    X_seq, y_seq = create_seq2seq_data(X_train, y_train, p['window_size'], p['n_ahead'])

    # Arquitectura Seq2Seq
    encoder_inputs = Input(shape=(p['window_size'], X_seq.shape[2]))
    encoder = LSTM(p['units'], return_sequences=False)(encoder_inputs)
    decoder = RepeatVector(p['n_ahead'])(encoder)
    decoder = LSTM(p['units'], return_sequences=True)(decoder)
    outputs = TimeDistributed(Dense(1))(decoder)

    model = Model(encoder_inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=p['learning_rate']), loss='mse')
    model.fit(X_seq, y_seq, epochs=p['epochs'], batch_size=p['batch_size'], verbose=0)

    # Construir secuencias para predicción
    X_full = np.vstack([X_train, X_test])
    start = len(X_train)
    end = len(X_full) - p['n_ahead'] + 1

    if end <= start:
        raise ValueError("Test set muy corto para generar predicciones con los parámetros actuales.")

    X_pred_seq = np.array([
        X_full[i - p['window_size']:i] for i in range(start, end)
    ])

    # Predicción y desescalado
    y_pred_scaled = model.predict(X_pred_seq, verbose=0)
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

    # Alineación segura de índice
    num_preds = len(y_pred)
    index_pred = test_set.index[-num_preds:] if num_preds <= len(test_set) else test_set.index

    return model, pd.Series(y_pred, index=index_pred, name='LSTM_Seq2Seq'), (scaler_x, scaler_y)

In [24]:
import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error

def optimize_seq2seq_lstm_cv(df, split, n_trials=10):
    """
    Optimiza hiperparámetros de un modelo Seq2Seq LSTM multivariado usando validación cruzada.
    """
    model_dict = {}

    def objective(trial):
        model_params = {
            'window_size': trial.suggest_int('window_size', 6, 24),
            'n_ahead': trial.suggest_int('n_ahead', 3, 12),
            'units': trial.suggest_int('units', 32, 128),
            'epochs': trial.suggest_int('epochs', 20, 100),
            'batch_size': trial.suggest_categorical('batch_size', [8, 16, 32]),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
        }

        mape_scores = []
        best_mape = np.inf

        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]

                # Asegura que el test_set tenga al menos n_ahead muestras
                if len(test_set) < model_params['n_ahead']:
                    continue

                _, y_pred, _ = fit_predict_eval_seq2seq_lstm(training_set, test_set, model_params)

                # Alinear longitud para evitar errores
                y_true = test_set.loc[y_pred.index, 'y']
                mape = mean_absolute_percentage_error(y_true, y_pred)
                mape_scores.append(mape)
                best_mape = min(best_mape, mape)

            if len(mape_scores) == 0:
                return np.inf

            model_dict[trial.number] = {
                'params': model_params,
                'mape_best': best_mape
            }

            return np.mean(mape_scores)

        except Exception as e:
            print(f"[ERROR] Seq2Seq Trial {trial.number} → {model_params} | {e}")
            return np.inf

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    trials_data = [
        (
            trial.number,
            trial.params,
            trial.value,
            model_dict.get(trial.number, {}).get('mape_best', np.inf)
        )
        for trial in study.trials
    ]

    trials_df = pd.DataFrame(
        trials_data, columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )

    return study.best_params, trials_df, model_dict

In [25]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.layers import GRU

from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention
from tensorflow.keras.models import Model


def fit_predict_eval_ckpt_model(training_set, test_set, ckpt_path, model_params=None):
    """
    Carga un modelo desde un checkpoint (.ckpt) y evalúa sobre datos de series de tiempo multivariados.
    """
    default_params = {
        'window_size': 12,
        'units': 50,  # Debe coincidir con los usados para entrenar el .ckpt
    }
    if model_params:
        default_params.update(model_params)
    p = default_params

    features = training_set.drop(columns=['ds', 'y']).columns
    scaler_x = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_train = scaler_x.fit_transform(training_set[features])
    y_train = scaler_y.fit_transform(training_set[['y']])
    X_test = scaler_x.transform(test_set[features])
    y_test = scaler_y.transform(test_set[['y']])

    # Preparar entrada para predicción (no se entrena)
    X_full = np.vstack([X_train, X_test])
    X_pred_seq = np.array([
        X_full[i - p['window_size']:i]
        for i in range(len(training_set), len(training_set) + len(test_set))
    ])

    # Construir la arquitectura (debe coincidir con la usada para entrenar el ckpt)
    model = Sequential([
        GRU(p['units'], return_sequences=True, input_shape=(p['window_size'], X_pred_seq.shape[2])),
        Dropout(0.2),
        GRU(p['units'] // 2),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer=Adam(), loss='mse')

    # Cargar pesos desde checkpoint
    model.load_weights(ckpt_path).expect_partial()

    # Predecir
    y_pred_scaled = model.predict(X_pred_seq, verbose=0)
    y_pred = scaler_y.inverse_transform(y_pred_scaled).flatten()

    return model, pd.Series(y_pred, index=test_set.index, name='CKPT_Model'), (scaler_x, scaler_y)

In [26]:
import optuna
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error

def optimize_ckpt_model_cv(df, split, ckpt_path, n_trials=1):
    """
    Evalúa un modelo preentrenado (.ckpt) usando validación cruzada.
    No entrena, solo carga y evalúa.
    """
    model_dict = {}

    def objective(trial):
        # Se pueden ajustar parámetros estructurales, pero deben coincidir con los del .ckpt
        model_params = {
            'window_size': trial.suggest_int('window_size', 6, 24),
            'units': trial.suggest_int('units', 16, 128)
        }

        mape_scores = []
        best_mape = np.inf

        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]

                _, y_pred, _ = fit_predict_eval_ckpt_model(training_set, test_set, ckpt_path, model_params)

                y_true = test_set.loc[y_pred.index, 'y']
                mape = mean_absolute_percentage_error(y_true, y_pred)

                mape_scores.append(mape)
                best_mape = min(best_mape, mape)

            model_dict[trial.number] = {
                'params': model_params,
                'mape_best': best_mape
            }

            return np.mean(mape_scores)

        except Exception as e:
            print(f"[ERROR] CKPT Trial {trial.number} → {model_params} | {e}")
            return np.inf

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    trials_data = [
        (
            trial.number,
            trial.params,
            trial.value,
            model_dict.get(trial.number, {}).get('mape_best', np.inf)
        )
        for trial in study.trials
    ]

    trials_df = pd.DataFrame(trials_data, columns=['trial_number', 'params', 'mape_mean', 'mape_best'])

    return study.best_params, trials_df, model_dict

In [23]:
pip install git+https://github.com/google-research/timesfm.git

  Cloning https://github.com/google-research/timesfm.git to /private/var/folders/_2/b07hnlxx27j566j7xpz5w05m0000gn/T/pip-req-build-vuhc8siq
  Running command git clone --filter=blob:none --quiet https://github.com/google-research/timesfm.git /private/var/folders/_2/b07hnlxx27j566j7xpz5w05m0000gn/T/pip-req-build-vuhc8siq
  Resolved https://github.com/google-research/timesfm.git to commit 5950ef0653e4ff142a80906a08432cc457a2ace9
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 97.9 MB/s eta 0:00:00
  Created wheel for timesfm: filename=timesfm-1.2.9-py3-none-any.whl size=54967 sha256=9b5a7411614cc100e5df1832388721c05c5488a95e87e1ffef12ef8a77b774e8
  Stored in directory: /private/var/folders/_2/b07hnlxx27j566j7xpz5w05m0000gn/T/pip-ephem-wheel-cache-svykxea5/wheels/aa/0e/b1

In [22]:
best_params_trans, trials_df_trans, model_dict_trans = optimize_ckpt_model_cv(df_final.assign(ds=df_final.index), splits, "./nbeats_m4_monthly.ckpt", 2)

/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[ERROR] CKPT Trial 0 → {'window_size': 20, 'units': 22} | File format not supported: filepath=./nbeats_m4_monthly.ckpt. Keras 3 only supports V3 `.keras` and `.weights.h5` files, or legacy V1/V2 `.h5` files.
[ERROR] CKPT Trial 1 → {'window_size': 6, 'units': 107} | File format not supported: filepath=./nbeats_m4_monthly.ckpt. Keras 3 only supports V3 `.keras` and `.weights.h5` files, or legacy V1/V2 `.h5` files.


In [4]:
pip uninstall -y numpy


Found existing installation: numpy 2.2.6
Uninstalling numpy-2.2.6:
  Successfully uninstalled numpy-2.2.6
Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install numpy==1.26.0

  Using cached numpy-1.26.0-cp310-cp310-macosx_11_0_arm64.whl.metadata (53 kB)
Using cached numpy-1.26.0-cp310-cp310-macosx_11_0_arm64.whl (14.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gluonts 0.14.4 requires pandas<2.2.0,>=1.0, but you have pandas 2.3.0 which is incompatible.
greykite 1.1.0 requires holidays==0.13, but you have holidays 0.24 which is incompatible.
greykite 1.1.0 requires pandas<2.0.0,>=1.5.0, but you have pandas 2.3.0 which is incompatible.
greykite 1.1.0 requires scikit-learn==1.3.1, but you have scikit-learn 1.5.2 which is incompatible.
greykite 1.1.0 requires scipy<1.15.2,>=1.15.0, but you have scipy 1.13.1 which is incompatible.
prophet 1.1.6 requires holidays<1,>=0.25, but you h

In [27]:
%%capture
from optimizers import optimize_hw_cv
#from optimizers import optimize_autoarima_cv
#from optimizers import optimize_silverkite_cv
from optimizers import optimize_lstm_cv
#from optimizers import optimize_gru_cv
#from optimizers import optimize_transformer_cv
#from optimizers import optimize_elastic_net_cv
from optimizers import optimize_prophet_cv
#from optimizers import optimize_mlp_cv
#from optimizers import optimize_transformer_cv
#from optimizers import optimize_tree_cv
#from optimizers import optimize_rf_cv
n_trials=50
best_params_hw, trials_df_hw, model_dict_hw = optimize_hw_cv(df_final, splits, n_trials)
trials_df_hw['model'] = 'holt_winters'
#best_params_arima, trials_df_arima, model_dict_arima = optimize_autoarima_cv(df_final, splits, n_trials)
#trials_df_arima['model'] = 'auto_arima'
#best_params_sk, trials_df_sk, model_dict_sk = optimize_silverkite_cv(df_final, splits, n_trials)
#trials_df_sk['model'] = 'silverkite'
best_params_lstm, trials_df_lstm, model_dict_lstm = optimize_lstm_cv(df_final.assign(ds=df_final.index), splits, n_trials)
trials_df_lstm['model'] = 'lstm'
#best_params_gru, trials_df_gru, model_dict_gru = optimize_gru_cv(df_final.assign(ds=df_final.index), splits, n_trials)
#trials_df_gru['model'] = 'gru'
#best_params_trans, trials_df_trans, model_dict_trans = optimize_transformer_cv(df_final.assign(ds=df_final.index), splits, n_trials)
#trials_df_trans['model'] = 'transformer'
#best_params_enet, trials_df_enet, model_dict_enet = optimize_elastic_net_cv(df_final.assign(ds=df_final.index),splits,n_trials)
#trials_df_enet['model'] = 'elastic_net'
best_params_prophet, trials_df_prophet, model_dict_prophet = optimize_prophet_cv(df_final.reset_index(), splits,n_trials)
trials_df_prophet['model'] = 'prophet'
best_params_mlp, trials_df_mlp, model_dict_mlp = optimize_mlp_cv(df_final.assign(ds=df_final.index), splits, n_trials)
trials_df_mlp['model'] = 'MLP'
#best_params_transformer, trials_df_transformer, model_dict_transformer = optimize_transformer_cv(df_final.assign(ds=df_final.index), splits, n_trials)
#trials_df_transformer['model'] = 'Transformer'
#best_params_tree, trials_df_tree, model_dict_tree = optimize_tree_cv(df_final.assign(ds=df_final.index), splits, n_trials)
#trials_df_tree['model'] = 'DecisionTree'
#best_params_rf, trials_df_rf, model_dict_rf = optimize_rf_cv(df_final.assign(ds=df_final.index), splits, n_trials)
#trials_df_rf['model'] = 'RandomForest'

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

NameError: name 'optimize_mlp_cv' is not defined

In [8]:
import pandas as pd
import pickle
results = {}
results['trials'] = pd.concat([
    trials_df_hw.assign(model='HW'),
    trials_df_arima.assign(model='SARIMAX'),
    trials_df_sk.assign(model='Silverkite'),
    trials_df_prophet.assign(model='Prophet'),
    trials_df_mlp.assign(model='MLP'),
    trials_df_gru.assign(model='GRU'),
    trials_df_lstm.assign(model='LSTM'),
    trials_df_enet.assign(model='ElasticNet'),
    trials_df_transformer.assign(model='Transformer'),
    trials_df_tree.assign(model='RegresionTree'),
    trials_df_rf.assign(model='RandomForest'),
], ignore_index=True)

results['features']=feat
with open('results'+name+'.pkl', 'wb') as file:
    pickle.dump(results, file)

In [32]:
!pip install git+https://github.com/google-research/timesfm.git

  Cloning https://github.com/google-research/timesfm.git to /private/var/folders/_2/b07hnlxx27j566j7xpz5w05m0000gn/T/pip-req-build-2ktaetrt
  Running command git clone --filter=blob:none --quiet https://github.com/google-research/timesfm.git /private/var/folders/_2/b07hnlxx27j566j7xpz5w05m0000gn/T/pip-req-build-2ktaetrt
  Resolved https://github.com/google-research/timesfm.git to commit 5950ef0653e4ff142a80906a08432cc457a2ace9
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [5]:
from sklearn.preprocessing import MinMaxScaler
from timesfm import TimesFm
import numpy as np
import pandas as pd

# Para evitar múltiples cargas del modelo
_timesfm_model = None

def fit_predict_eval_timesfm(training_set, test_set, model_params=None):
    """
    Evalúa el modelo preentrenado TimesFM sobre datos de series de tiempo univariadas.
    """
    global _timesfm_model

    default_params = {
        'context_length': 128,
        'prediction_length': 24
    }
    if model_params:
        default_params.update(model_params)
    p = default_params

    # Solo se usa la variable target 'y'
    scaler_y = MinMaxScaler()
    y_train = scaler_y.fit_transform(training_set[['y']]).flatten()
    y_test = scaler_y.transform(test_set[['y']]).flatten()

    y_all = np.concatenate([y_train, y_test])

    total_required_length = p['context_length'] + p['prediction_length']
    if len(y_all) < total_required_length:
        raise ValueError("No hay suficientes datos para el contexto y predicción requeridos por TimesFM.")

    # Inicializa el modelo una vez
    if _timesfm_model is None:
        _timesfm_model = TimesFm.from_pretrained("google-research/timesfm-1.0")

    # Preparar input para predicción
    input_series = y_all[-total_required_length:-p['prediction_length']]
    input_tensor = tf.convert_to_tensor(input_series.reshape(1, -1), dtype=tf.float32)

    # Predicción
    y_pred_scaled = _timesfm_model(input_tensor).numpy().flatten()[:p['prediction_length']]
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

    # Índices alineados con test_set
    index_pred = test_set.index[:len(y_pred)]
    return _timesfm_model, pd.Series(y_pred, index=index_pred, name='TimesFM'), scaler_y

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.
Loaded PyTorch TimesFM, likely because python version is 3.10.18 (main, Jun  3 2025, 18:23:41) [Clang 17.0.0 (clang-1700.0.13.3)].


In [6]:
import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error

def optimize_timesfm_cv(df, split, n_trials=10):
    """
    Optimiza la longitud de contexto y predicción de TimesFM usando validación cruzada.
    """

    model_dict = {}

    def objective(trial):
        model_params = {
'context_length': trial.suggest_int('context_length', 24, 72),   # 2 a 6 años de contexto
'prediction_length': trial.suggest_int('prediction_length', 6, 18)  # 6 meses a 1.5 años
        }

        mape_scores = []
        best_mape = np.inf

        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]

                # Para evitar errores si el set es más corto que lo requerido
                if len(training_set) + len(test_set) < model_params['context_length'] + model_params['prediction_length']:
                    raise ValueError("No hay suficientes datos para el contexto requerido por TimesFM.")

                _, y_pred, _ = fit_predict_eval_timesfm(training_set, test_set, model_params)
                y_true = test_set.loc[y_pred.index, 'y']
                mape = mean_absolute_percentage_error(y_true, y_pred)

                mape_scores.append(mape)
                best_mape = min(best_mape, mape)

            model_dict[trial.number] = {
                'params': model_params,
                'mape_best': best_mape
            }

            return np.mean(mape_scores)

        except Exception as e:
            print(f"[ERROR] TimesFM Trial {trial.number} → {model_params} | {e}")
            return np.inf

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    trials_data = [
        (
            trial.number,
            trial.params,
            trial.value,
            model_dict.get(trial.number, {}).get('mape_best', np.inf)
        )
        for trial in study.trials
    ]

    trials_df = pd.DataFrame(
        trials_data, columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )

    return study.best_params, trials_df, model_dict

In [4]:
df_final


,y,imacec_no_minero_lag_0,imacec_lag_0,consumo_hogar_ipsfl_lag_0,bienos_no_durables_lag_0,al_lag_0,cu_lag_1,tasa_int_prom_vivienda_lag_0,movilidad_lag_0,zn_lag_0,ExtremoSur_tavg_lag_0,ipc_sin_volatiles_mensual_lag_0,hierro_fob_lag_4,molibdeno_fob_lag_0,Primavera_lag_1
ds,,,,,,,,,,,,,,,
2017-05-01,4779.257225,97.280222,96.953323,27418.298860,11783.945565,0.874660,2.6810,3.36,0.0,1.179673,6.253226,0.105915,99.505775,16.540103,0.0
2017-06-01,4075.594295,94.470379,94.127715,27676.218700,11769.215395,0.870917,2.6575,3.29,0.0,1.253630,2.670000,0.020346,68.453212,5.901190,0.0
2017-07-01,4328.359903,90.578043,90.907381,27838.825325,11766.129725,0.868761,2.7690,3.20,0.0,1.269510,4.682258,0.102782,91.434381,11.917892,0.0
2017-08-01,4899.666800,95.377361,95.779130,27906.118734,11774.688555,0.960073,2.9775,3.19,0.0,1.433530,5.048387,0.096109,74.299944,23.455658,0.0
2017-09-01,4535.573688,92.260679,92.872064,27878.098928,11794.891883,0.952586,3.1650,3.20,0.0,1.436706,6.861667,-0.027008,60.546277,15.481996,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-01,5828.768795,111.685315,109.512246,31876.078818,12979.360750,1.062840,4.1600,4.97,0.0,1.130469,5.011290,0.096739,181.587180,23.865209,0.0
2024-09-01,5207.977220,106.938772,105.270940,31796.265131,12986.180026,1.149841,4.1450,4.81,0.0,1.142813,5.691667,0.339036,90.498837,13.509175,0.0
2024-10-01,6855.941037,112.986265,110.577523,31796.265131,12986.180026,1.161751,4.4960,4.54,0.0,1.104688,10.109677,0.412874,142.286599,49.409920,0.5


In [7]:
df_timefm = df_final.copy()
#df_final['ds'] = pd.to_datetime(df_final['ds'])  # si no está ya
df_timefm = df_timefm.sort_values('ds').reset_index(drop=True)
best_params, trials_df, model_dict = optimize_timesfm_cv(df_timefm, splits, n_trials=5)

[ERROR] TimesFM Trial 0 → {'context_length': 58, 'prediction_length': 17} | type object 'TimesFmTorch' has no attribute 'from_pretrained'
[ERROR] TimesFM Trial 1 → {'context_length': 63, 'prediction_length': 12} | type object 'TimesFmTorch' has no attribute 'from_pretrained'
[ERROR] TimesFM Trial 2 → {'context_length': 31, 'prediction_length': 6} | type object 'TimesFmTorch' has no attribute 'from_pretrained'
[ERROR] TimesFM Trial 3 → {'context_length': 68, 'prediction_length': 6} | type object 'TimesFmTorch' has no attribute 'from_pretrained'
[ERROR] TimesFM Trial 4 → {'context_length': 31, 'prediction_length': 7} | type object 'TimesFmTorch' has no attribute 'from_pretrained'


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
import pandas as pd
import torch
_llm_model = None
_tokenizer = None
def fit_predict_eval_llm_forecaster(training_set, test_set, model_params=None):
    """
    Simula la predicción de series de tiempo usando un LLM como si fuera un modelo de forecasting.
    Transforma la serie temporal a texto, el modelo predice tokens que representan valores futuros.
    """
    global _llm_model, _tokenizer

    default_params = {
        'context_length': 128,
        'prediction_length': 24,
        'model_name': 'gpt2'  # o uno de huggingface como mistralai/Mistral-7B-Instruct-v0.2
    }
    if model_params:
        default_params.update(model_params)
    p = default_params

    scaler_y = MinMaxScaler()
    y_train = scaler_y.fit_transform(training_set[['y']]).flatten()
    y_test = scaler_y.transform(test_set[['y']]).flatten()
    y_all = np.concatenate([y_train, y_test])

    total_required_length = p['context_length'] + p['prediction_length']
    if len(y_all) < total_required_length:
        raise ValueError("No hay suficientes datos para el contexto y predicción requeridos.")

    # Inicializa el modelo y tokenizer
    if _llm_model is None or _tokenizer is None:
        _tokenizer = AutoTokenizer.from_pretrained(p['model_name'])
        _llm_model = AutoModelForCausalLM.from_pretrained(p['model_name'])

    # Codifica la serie como texto (e.g., coma separada)
    context_series = y_all[-total_required_length:-p['prediction_length']]
    context_str = ", ".join([f"{v:.3f}" for v in context_series])
    prompt = f"Given the previous values: [{context_str}], predict the next {p['prediction_length']} values:"

    inputs = _tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = _llm_model.generate(**inputs, max_new_tokens=50, pad_token_id=_tokenizer.eos_token_id)

    decoded = _tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extraer los números predichos desde el texto (simple parsing, no robusto)
    import re
    predicted_str = decoded.split("predict the next")[1]
    predicted_values = re.findall(r"\d+\.\d+", predicted_str)
    y_pred_scaled = np.array(predicted_values[:p['prediction_length']], dtype=float)

    # Inversa del escalamiento
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    index_pred = test_set.index[:len(y_pred)]
    return _llm_model, pd.Series(y_pred, index=index_pred, name='LLMForecast'), scaler_y

In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error

def optimize_llm_forecaster_cv(df, split, n_trials=10):
    """
    Optimiza la longitud de contexto y predicción de un modelo LLM para series de tiempo usando validación cruzada.
    """
    model_dict = {}

    def objective(trial):
        model_params = {
            'context_length': trial.suggest_int('context_length', 24, 72),  # 2 a 6 años (mensual)
            'prediction_length': trial.suggest_int('prediction_length', 6, 18),  # 6 a 18 meses
            'model_name': 'gpt2'  # puedes probar otros como 'tiiuae/falcon-rw-1b' o 'mistralai/Mistral-7B-Instruct-v0.2'
        }

        mape_scores = []
        best_mape = np.inf

        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]

                total_len = len(training_set) + len(test_set)
                required_len = model_params['context_length'] + model_params['prediction_length']

                if total_len < required_len:
                    raise ValueError("No hay suficientes datos para el contexto y predicción requeridos por el modelo.")

                _, y_pred, _ = fit_predict_eval_llm_forecaster(training_set, test_set, model_params)
                y_true = test_set.loc[y_pred.index, 'y']
                mape = mean_absolute_percentage_error(y_true, y_pred)

                mape_scores.append(mape)
                best_mape = min(best_mape, mape)

            model_dict[trial.number] = {
                'params': model_params,
                'mape_best': best_mape
            }

            return np.mean(mape_scores)

        except Exception as e:
            print(f"[ERROR] LLM Trial {trial.number} → {model_params} | {e}")
            return np.inf

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    trials_data = [
        (
            trial.number,
            trial.params,
            trial.value,
            model_dict.get(trial.number, {}).get('mape_best', np.inf)
        )
        for trial in study.trials
    ]

    trials_df = pd.DataFrame(
        trials_data, columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )

    return study.best_params, trials_df, model_dict

/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [15]:
model_dict

{0: {'params': {'context_length': 25,
   'prediction_length': 13,
   'model_name': 'gpt2'},
  'mape_best': 0.06881717961250186},
 1: {'params': {'context_length': 61,
   'prediction_length': 9,
   'model_name': 'gpt2'},
  'mape_best': 0.07591123034625144},
 2: {'params': {'context_length': 41,
   'prediction_length': 15,
   'model_name': 'gpt2'},
  'mape_best': 0.11163327928334794},
 3: {'params': {'context_length': 65,
   'prediction_length': 17,
   'model_name': 'gpt2'},
  'mape_best': 0.0820939446229937},
 4: {'params': {'context_length': 56,
   'prediction_length': 7,
   'model_name': 'gpt2'},
  'mape_best': 0.12055316520764754}}

In [14]:
trials_df

,trial_number,params,mape_mean,mape_best
0,0,"{'context_length': 25, 'prediction_length': 13}",0.096509,0.068817
1,1,"{'context_length': 61, 'prediction_length': 9}",0.109134,0.075911
2,2,"{'context_length': 41, 'prediction_length': 15}",0.270591,0.111633
3,3,"{'context_length': 65, 'prediction_length': 17}",0.119934,0.082094
4,4,"{'context_length': 56, 'prediction_length': 7}",0.168497,0.120553


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from llama_cpp import Llama
import numpy as np
import pandas as pd
import re

_llm_gguf_model = None  # Global para evitar recarga

def fit_predict_eval_llm_forecaster_gguf(training_set, test_set, model_params=None):
    """
    Simula la predicción de series de tiempo usando un LLM en formato .gguf como modelo de forecasting.
    Convierte la serie a texto, genera predicciones y las transforma a números.
    """
    global _llm_gguf_model

    default_params = {
        'context_length': 128,
        'prediction_length': 24,
        'model_path': '/ruta/a/modelo.gguf',  # ← CAMBIA ESTO por el path real a tu archivo .gguf
        'max_tokens': 128,
        'n_ctx': 2048
    }
    if model_params:
        default_params.update(model_params)
    p = default_params

    scaler_y = MinMaxScaler()
    y_train = scaler_y.fit_transform(training_set[['y']]).flatten()
    y_test = scaler_y.transform(test_set[['y']]).flatten()
    y_all = np.concatenate([y_train, y_test])

    total_required_length = p['context_length'] + p['prediction_length']
    if len(y_all) < total_required_length:
        raise ValueError("No hay suficientes datos para el contexto y predicción requeridos.")

    # Cargar modelo .gguf solo una vez
    if _llm_gguf_model is None:
        _llm_gguf_model = Llama(model_path=p['model_path'], n_ctx=p['n_ctx'])

    # Generar el prompt con valores normalizados
    context_series = y_all[-total_required_length:-p['prediction_length']]
    context_str = ", ".join([f"{v:.3f}" for v in context_series])
    prompt = f"Given the previous values: [{context_str}], predict the next {p['prediction_length']} values:"

    # Inferencia
    output = _llm_gguf_model(prompt, max_tokens=p['max_tokens'], stop=["\n"], echo=False)
    generated_text = output['choices'][0]['text']

    # Extraer números predichos del texto
    predicted_values = re.findall(r"\d+\.\d+", generated_text)
    if len(predicted_values) < p['prediction_length']:
        raise ValueError(f"El modelo solo generó {len(predicted_values)} valores, se esperaban {p['prediction_length']}.")

    y_pred_scaled = np.array(predicted_values[:p['prediction_length']], dtype=float)

    # Desnormalizar
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    index_pred = test_set.index[:len(y_pred)]

    return _llm_gguf_model, pd.Series(y_pred, index=index_pred, name='LLMForecast_GGUF'), scaler_y

In [16]:
pip install huggingface-hub

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
!huggingface-cli download TheBloke/Llama-2-7B-GGUF llama-2-7b.Q4_K_M.gguf --local-dir models/

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


llama-2-7b.Q4_K_M.gguf: 100%|███████████████| 4.08G/4.08G [00:37<00:00, 109MB/s]
Download complete. Moving file to models/llama-2-7b.Q4_K_M.gguf
models/llama-2-7b.Q4_K_M.gguf


In [19]:
from sklearn.preprocessing import MinMaxScaler
from llama_cpp import Llama
import numpy as np
import pandas as pd
import re

_llm_gguf_model = None  # Global para evitar recarga

def fit_predict_eval_llm_forecaster_gguf(training_set, test_set, model_params=None):
    """
    Simula la predicción de series de tiempo usando un LLM en formato .gguf como modelo de forecasting.
    Convierte la serie a texto, genera predicciones y las transforma a números.
    """
    global _llm_gguf_model

    default_params = {
        'context_length': 128,
        'prediction_length': 24,
        'model_path': './models/llama-2-7b.Q4_K_M.gguf',  # ← CAMBIA ESTO por el path real a tu archivo .gguf
        'max_tokens': 128,
        'n_ctx': 2048
    }
    if model_params:
        default_params.update(model_params)
    p = default_params

    scaler_y = MinMaxScaler()
    y_train = scaler_y.fit_transform(training_set[['y']]).flatten()
    y_test = scaler_y.transform(test_set[['y']]).flatten()
    y_all = np.concatenate([y_train, y_test])

    total_required_length = p['context_length'] + p['prediction_length']
    if len(y_all) < total_required_length:
        raise ValueError("No hay suficientes datos para el contexto y predicción requeridos.")

    # Cargar modelo .gguf solo una vez
    if _llm_gguf_model is None:
        _llm_gguf_model = Llama(model_path=p['model_path'], n_ctx=p['n_ctx'])

    # Generar el prompt con valores normalizados
    context_series = y_all[-total_required_length:-p['prediction_length']]
    context_str = ", ".join([f"{v:.3f}" for v in context_series])
    prompt = f"Given the previous values: [{context_str}], predict the next {p['prediction_length']} values:"

    # Inferencia
    output = _llm_gguf_model(prompt, max_tokens=p['max_tokens'], stop=["\n"], echo=False)
    generated_text = output['choices'][0]['text']

    # Extraer números predichos del texto
    predicted_values = re.findall(r"\d+\.\d+", generated_text)
    if len(predicted_values) < p['prediction_length']:
        raise ValueError(f"El modelo solo generó {len(predicted_values)} valores, se esperaban {p['prediction_length']}.")

    y_pred_scaled = np.array(predicted_values[:p['prediction_length']], dtype=float)

    # Desnormalizar
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    index_pred = test_set.index[:len(y_pred)]

    return _llm_gguf_model, pd.Series(y_pred, index=index_pred, name='LLMForecast_GGUF'), scaler_y

In [20]:
import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error

def optimize_llm_forecaster_cv_gguf(df, split, n_trials=10):
    """
    Optimizador de hiperparámetros para forecasting con LLM .gguf usando Optuna y validación cruzada.
    """
    from sklearn.exceptions import NotFittedError

    model_dict = {}

    def objective(trial):
        model_params = {
            'context_length': trial.suggest_int('context_length', 24, 72),
            'prediction_length': trial.suggest_int('prediction_length', 6, 18),
            'model_path': './models/llama-2-7b.Q4_K_M.gguf',
            'max_tokens': 128,
            'n_ctx': 2048
        }

        mape_scores = []
        best_mape = np.inf

        try:
            for fold, (train_index, test_index) in enumerate(split):
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]

                total_len = len(training_set) + len(test_set)
                required_len = model_params['context_length'] + model_params['prediction_length']

                if total_len < required_len:
                    raise ValueError("No hay suficientes datos para el contexto y predicción requeridos.")

                _, y_pred, _ = fit_predict_eval_llm_forecaster_gguf(training_set, test_set, model_params)
                y_true = test_set.loc[y_pred.index, 'y']

                mape = mean_absolute_percentage_error(y_true, y_pred)
                mape_scores.append(mape)
                best_mape = min(best_mape, mape)

            model_dict[trial.number] = {
                'params': model_params,
                'mape_best': best_mape
            }

            return np.mean(mape_scores)

        except Exception as e:
            print(f"[ERROR] Trial {trial.number} → {model_params} | {e}")
            return np.inf

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    trials_data = [
        (
            trial.number,
            trial.params,
            trial.value,
            model_dict.get(trial.number, {}).get('mape_best', np.inf)
        )
        for trial in study.trials
    ]

    trials_df = pd.DataFrame(
        trials_data, columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )

    return study.best_params, trials_df, model_dict